# Assignment #1: Pseudonymisation Techniques and Considerations
- Dataset: Crossfit [Daset](https://data.world/bgadoci/crossfit-data) (In this assignment only the athletes file was used) 
- Credits: Dataset was put together by Sam Swift
- ToDo: To run the jupyter notebook the requirements.txt need be installed (`pip install -r requirements.txt`)

In [ ]:
import pandas as pd

# Read csv as dataframe
dataframe = pd.read_csv("athletes.csv")

## 3.1 Pseudonymisation
Goals:
-  Determine which attributes qualify as explicit personally identifiable information (3.1.1)
    - Why? 
    - What method was used to identify these? 
- Generate pseudonymous values for the identified attributes (3.1.2)

### 3.1.1 Identifying Attributes containing Personally Identifiable Information (PII)
- In order to identify th attributes first a better understanding of the structure of the dataset needs to be obtained
    - What columns does the dataset contain and in what format are the attribute values?
        - Therefore, each column and the first value of each column (which is not empty or Null) is printed

- By inspecting the different columns and the data format, several attributes which have the potential to contain explicit personally identifiable information can be identified
    - `athelete_id`
        -  This really depends on the usage of this id! Considerations to take into account are: 
            - Is the `athlete_id` only used as an internal id of this dataset or does it maybe even refer to an official id?
            - Are there other datasets available which may have a similar source to this dataset? Thus, these other datasets may use the same `athlete_id`
    - `name`
        - The name allows to identify an individual
    - `team`
        - Depending on the size of the team, this could allow to identify a specific athlete
    - `affiliate` 
        - Depending on the affiliate and the amount of contracted athletes, this could allow to identify an individual
    - All stats of the athletes
        - If an athlete has really remarkable stats (maybe even a world record in a category), this could allow to identify the individual
    - `train` 
        - If an athlete has a special and famous training routine, this could allow to identify him
    - `background`
        - If an athlete has a famous background or mentions names, this could allow to identify him
    - `experience`
        - If an athlete mentions concrete information about his experience (e.g. name of current coach), this could allow to identify him

-> As can be seen, all columns could potentially contain outliers which could be then used to identify an individual. As agreed on, for each of the columns (only those holding numbers or names) potentially leaking PII one anonymization technique was chosen which keeps the data loss as minimal as possible without leaking obvious PII. It is important to note, that there could still be columns containing PII. However, this chance for outliers was accepted in this assignment. Columns containing descriptions (e.g. train, background and experience) where to anonymized and thus the chance for outliers accepted.

### 3.1.2 Pseudonymizing
- For pseudonymisation especially the name is suitable, because it can be easily replaced with a fake name, without being as obvious or destroying the purpose of the dataset
- With the help of the `anonymizedf` library new fake names can be created for the `name` column

In [ ]:
from anonymizedf import anonymizedf

# Prepare the data to be anonymized
an = anonymizedf.anonymize(dataframe)

In [ ]:
# Add new column with fake name
an.fake_names("name")

# Drop old original name column
dataframe = dataframe.drop(columns=['name'])

# Rename Fake_name column in name
dataframe = dataframe.rename(columns={'Fake_name': 'name'})

print(dataframe.iloc[:4])

## 3.2 Randomisation
Goal:
- Use randomisation technique to generate random strings (no meaning) (3.2.1)
- Use randomisation technique to generate random but meaningful replacements (3.2.2)
- Create a lookup table to keep track of the changes (3.2.3)

### 3.2.1 Random Strings
1. Identify columns to replace with random strings
    - Here columns containing words can be used 
        - `name`
        - `region`
        - `team`
        - `affiliate` 
2. Generate a function which is responsible for generating random values for replacement
3. Overwrite each value in respective attributes by using the generated value

In [ ]:
def get_first_not_not_empty_value(df_column):
    return df_column.dropna().iloc[0] if not df_column.dropna().empty else None

# Iterate each column 
for column in dataframe.columns:
    first_value = get_first_not_not_empty_value(dataframe[column])
    print(f"Column: '{column}', Example Data: {first_value}")

In [ ]:
import random
import string

# Define function to create random string
def generate_random_string(length):
    letters = string.ascii_letters
    return ''.join(random.choice(letters) for _ in range(length))

# Apply randomization on specified columns
dataframe['name'] = dataframe['name'].apply(lambda x: generate_random_string(10))
dataframe['region'] = dataframe['region'].apply(lambda x: generate_random_string(15))
dataframe['team'] = dataframe['team'].apply(lambda x: generate_random_string(20))
dataframe['affiliate'] = dataframe['affiliate'].apply(lambda x: generate_random_string(25))

print(dataframe.iloc[:4])

In [ ]:
def get_first_not_not_empty_value(df_column):
    return df_column.dropna().iloc[0] if not df_column.dropna().empty else None

# Iterate each column 
for column in dataframe.columns:
    first_value = get_first_not_not_empty_value(dataframe[column])
    print(f"Column: '{column}', Example Data: {first_value}")

### 3.2.2 Randomization with meaningful strings
1. Identify columns suitable
    - name, region, team, affiliate
        -  For randomizing the name, the first letter will be kept equal to the original in order to keep similarities
2. Write functionality to randomize the values

Interchange Values in Columns Region, Team, Affiliate with meaningful strings to randomize and print out the lookuptable

In [ ]:
from anonymizedf import anonymizedf

# To see the changes properly we have to reload the initial data
dataframe = pd.read_csv("athletes.csv")

# Interchange values for region, team and affiliate
def interchange_values(column_name):
    dataframe[column_name] = dataframe[column_name].sample(frac=1).reset_index(drop=True)
        
interchange_values('region')
interchange_values('team')
interchange_values('affiliate')

print(dataframe.iloc[:4])

In [ ]:
# To see the changes properly we have to reload the initial data
dataframe = pd.read_csv("athletes.csv")

# Copy the original values in mapping dataframe
dataframe_mapping = dataframe[['name', 'region', 'team', 'affiliate']].copy()

# Randomize region, team and affiliate
interchange_values('region')
interchange_values('team')
interchange_values('affiliate')

# Copy the modified values in mapping dataframe
dataframe_mapping[['randomized_region', 'randomized_team', 'randomized_affiliate']] = dataframe[['region', 'team', 'affiliate']].copy()

print(dataframe_mapping.iloc[:4])

1. Synthetic names are generated using the Faker library. Functions are defined to create fake first and last names for each letter of the alphabet.
2. The 'name' column is split into 'first_name' and 'last_name' columns for targeted pseudonymization.
3. A loop iterates through the dataset, replacing original names with randomly chosen pseudonyms that match the initial letter of the original names.
4. A dictionary ('name_mapping') is created to store the mapping between the original and pseudonymized names for reference.
5. Pseudonymized names are integrated into the dataset, replacing the original names in the 'name' column. The updated dataset is saved to a new CSV file ('updated_athletes.csv').

In [ ]:
import string
import pandas as pd
from faker import Faker
import random
from itertools import chain

def generate_fake_first_names_with_letter(letter, num_names):
    fake = Faker()
    first_names = []

    while len(first_names) < num_names:
        try:
            fake_name = fake.unique.first_name()
            #print(fake_name)
            if fake_name.startswith(letter):
                first_names.append(fake_name)
        except:
            return first_names
    return first_names

def generate_fake_last_names_with_letter(letter, num_names):
    fake = Faker()
    last_names = []

    while len(last_names) < num_names:
        try:
            fake_name = fake.unique.last_name()
            #print(fake_name)
            if fake_name.startswith(letter):
                last_names.append(fake_name)
        except:
            return last_names
    return last_names


# Load the dataset
dataframe = pd.read_csv('athletes.csv')

# Create a dictionary to store the mapping from the original name to the pseudonymized name
fake = Faker()

first_names =[]
last_names = []

name_mapping = {}

# Function to generate a random string of a given length
for letter in string.ascii_uppercase:
    try:
        first_names.append(generate_fake_first_names_with_letter(letter, 10))
    except:
        continue

for letter in string.ascii_uppercase:
    try:
        last_names.append(generate_fake_last_names_with_letter(letter, 10))
    except:
        continue

first_names = list(chain(*first_names))
last_names = list(chain(*last_names))

dataframe[['first_name', 'last_name']] = dataframe['name'].str.split(' ', n=1, expand=True)

# Loop through the 'first_name' column and update the values
for i in range(len(dataframe)):
    try:
        first_letter_first_name = dataframe.at[i, 'first_name'][0]
        first_names_with_first_letter = [name for name in first_names if name.startswith(first_letter_first_name)]
        old_first_name = dataframe.at[i, 'first_name']
        dataframe.at[i, 'first_name'] = random.choice( first_names_with_first_letter)


        first_letter_last_name = dataframe.at[i, 'last_name'][0]
        old_last_name = dataframe.at[i, 'last_name']
        last_names_with_first_letter = [name for name in first_names if name.startswith(first_letter_last_name)]
        dataframe.at[i, 'last_name'] = random.choice(last_names_with_first_letter)

        old_name = old_first_name + " " + old_last_name
        new_name = dataframe.at[i,'first_name'] + ' ' + dataframe.at[i,'last_name']
        #print(old_name)
        #print(new_name)
        name_mapping[str(old_name)] = str(new_name)
    except:
        continue
print(name_mapping)
names_list = list(name_mapping.items())
dataframe = pd.DataFrame(names_list)

dataframe['name'] = dataframe['first_name'] + ' ' + dataframe['last_name']
df = dataframe.drop(['first_name', 'last_name'], axis=1)
#df.to_csv('athletes.csv', index=False)
#dataframe.to_csv('randomisation_lookup_table.csv', index=False)


## 3.3 Aggregation
Goal:
- Determine attributes which qualify for aggregation (3.3.1)
- Write a function to perform that aggregation process (3.3.2)

### 3.3.1 Determine Attributes for Aggregation

The script iterates through each column in the DataFrame and prints the column name along with the first non-empty value in that column:

In [ ]:
# Print each column with its first not empty value
for column in dataframe.columns:
    first_value = get_first_not_not_empty_value(dataframe[column])
    print(f"Column: '{column}', Example Data: {first_value}")

- Numerical attributes can be easily aggregated
- Especially those related to information about the individual are suitable for aggregation
    - `age`
    - `height` 
    - `weight`
- To be able to identify different levels of values, the extremes have to be identified 

The script prints the minimum and maximum values for 'age,' 'height,' and 'weight' columns:

In [ ]:
print(f"Minimum age': {dataframe['age'].nsmallest(5)}, Maximum age: {dataframe['age'].nlargest(5)}")
print(f"Minimum height': {dataframe['height'].nsmallest(10)}, Maximum height: {dataframe['height'].nlargest(10)}")
print(f"Minimum weight': {dataframe['weight'].nsmallest(10)}, Maximum weight: {dataframe['weight'].nlargest(10)}")

- Because even by observing ten extremes for the height and weight the values didn't make sense, in the following average values for the entire population were taken

### 3.3.1 Perform Aggregation

The script defines bins and labels for 'age,' 'height,' and 'weight' columns, categorizing the numerical values into bins. This provides a more concise representation of the data distribution:
Finally, the modified DataFrame, incorporating binning for 'age,' 'height,' and 'weight,' is printed:

In [ ]:
bins_age = [0, 18, 30, 45, 60, 100]
labels_age = ['0-18', '19-30', '31-45', '46-60', '60+']

bins_height = [0, 159, 169, 179, 189, 199, 220]
labels_height = ['0-159', '160-169', '170-179', '180-189', '190-199', '200+']

bins_weight = [0, 49, 59, 69, 79, 89, 99, 130]
labels_weight = ['0-49', '50-59', '60-69', '70-79', '80-89', '90-99', '100+']

dataframe['age'] = pd.cut(dataframe['age'], bins=bins_age, labels=labels_age)
dataframe['height'] = pd.cut(dataframe['height'], bins=bins_age, labels=labels_age)
dataframe['weight'] = pd.cut(dataframe['weight'], bins=bins_age, labels=labels_age)

print(dataframe)

## 3.4 Perturbation
Goal:
- Select attributes to add noise to (3.4.1)
- Implement the functionality to add the noise (3.4.2)
    - The original distribution of values should be preserved 
- Analyse distribution of original data and then with noise added (3.4.3)

### 3.4.1 Select attributes
- Similar to the aggregation, perturbation as well fits best for numbers as attribute values 
- Because the stats of the athletes are the main subject of the dataset they should normally be no noise added to them
    - Even more, the correctness of the value really matters in order to be able to compare different athletes. Here already a small noise is not good
- However, as already mentioned in 3.1 this kind of information loss is accepted in this assignment
- Therefore, the stats will be pertubated 

### 3.4.2 Pertubate Values
- Standard deviation was used to determine noise

The initial step involves loading the athlete dataset ('athletes.csv') using the pandas library:
Continuous variables to be perturbed are explicitly specified:
Statistical properties (mean and standard deviation) of non-missing numeric values in the selected columns are analyzed before perturbation:
Non-missing numeric values in each specified column are perturbed by adding random noise, determined by a noise factor (in this case, 40). The perturbed data is then rounded and updated in the DataFrame:
Statistical properties of non-missing numeric values are re-analyzed after perturbation:
The final step involves saving the perturbed data back to the original CSV file.

In [ ]:
import pandas as pd
import numpy as np

# Load data from the CSV file
dataframe = pd.read_csv('athletes.csv')

columns_to_perturb = ['pullups', 'helen', 'grace', 'filthy50', 'fgonebad', 'run400', 'run5k', 'candj', 'snatch', 'deadlift', 'backsq']

# Analyze the non-missing numeric values before perturbation
stats_before_perturbation = {}
for column in columns_to_perturb:
    data = dataframe[column].values
    non_missing_values = [value for value in data if not pd.isna(value)]
    mean_before = np.mean(non_missing_values)
    std_dev_before = np.std(non_missing_values)
    stats_before_perturbation[column] = {'mean_before': mean_before, 'std_dev_before': std_dev_before}

print("Before Perturbation:")
for column, stats in stats_before_perturbation.items():
    print(f"Mean {column} (Before): {stats['mean_before']}")
    print(f"Std Dev {column} (Before): {stats['std_dev_before']}")

# Choose a noise factor
noise_factor = 40  # Adjust this based on your privacy requirements

# Perturb the non-missing numeric values for each specified column
for column in columns_to_perturb:
    perturbed_data = []
    for value in dataframe[column]:
        if not pd.isna(value):
            perturbed_value = value + np.random.normal(0, noise_factor)
            perturbed_value = int(abs(round(perturbed_value)))
            perturbed_data.append(perturbed_value)
        else:
            perturbed_data.append(np.nan)
    dataframe[column] = perturbed_data  # Update the column in the DataFrame with perturbed data


# Save the perturbed data back to a CSV file without scientific notation
dataframe.to_csv('athletes.csv', index=False, float_format='%.0f')

### 3.4.3 Analyze after Pertubation

In [ ]:
import matplotlib.pyplot as plt

# Analyze the non-missing numeric values after perturbation
stats_after_perturbation = {}
for column in columns_to_perturb:
    non_missing_values_after = [value for value in dataframe[column] if not pd.isna(value)]
    mean_after = np.mean(non_missing_values_after)
    std_dev_after = np.std(non_missing_values_after)
    stats_after_perturbation[column] = {'mean_after': mean_after, 'std_dev_after': std_dev_after}

print("\nAfter Perturbation:")
for column, stats in stats_after_perturbation.items():
    print(f"Mean {column} (After): {stats['mean_after']}")
    print(f"Std Dev {column} (After): {stats['std_dev_after']}")
    

fig, (ax1,ax2) = plt.subplots(nrows=1, ncols=2, sharey=True, figsize=(20,10))
ax1.hist(
    'helen', 
    data=pd.read_csv('athletes.csv'),
    bins = np.arange(start=100, stop=2_000, step=200),
)
ax1.title.set_text('Original Distribution')
ax2.hist(
    'helen', 
    data=dataframe, 
    bins = np.arange(start=100, stop=2_000, step=200),
)
ax2.title.set_text('Distribution with noise added')

## 3.5 Data Analysis
Goal:
- Determine data loss using function (3.5.1)
- Discuss pros and cons (3.5.2)
- attributes included in the information loss function: ['pullups', 'helen', 'grace', 'filthy50', 'fgonebad', 'run400', 'run5k', 'candj', 'snatch', 'deadlift', 'backsq'] 
- maybe aggregated values: age, height, weight, but they are not continious variables, so they wont be respected in the following informatiin loss function


The initial step involves reading two CSV files, 'athletes.csv' and 'athletes_og.csv', using the pandas library, in which the athletes_og.csv is the original file and the athletes.csv is the file containing the perturbed continuous values.
Aggregated, randomized and pseudonimized are not continuous and are therefore not considered in the Data Analysis.
The Continuous variables to be analysed are explicitly specified.
A function named 'calculate_information_loss' is defined to calculate the information loss between two arrays using the mean squared differences and a scaling factor.
Information loss is calculated for each column, aggregated, and then the total information loss is printed:

In [ ]:
import pandas as pd
import numpy as np

# Read the CSV files

original_data = pd.read_csv("athletes.csv")

# Columns to calculate information loss
columns_to_compare = ['pullups', 'helen', 'grace', 'filthy50', 'fgonebad', 'run400', 'run5k', 'candj', 'snatch', 'deadlift', 'backsq']

# Calculate Information Loss
def calculate_information_loss(x, y, S):
    return (x - y) / np.sqrt(2 * S)

total_information_loss = 0
n = len(original_data)  # Assuming both datasets have the same number of rows
p = len(columns_to_compare)  # Number of columns to compare

for column in columns_to_compare:
    x = original_data[column]
    y = dataframe[column]
    S = np.mean((x - y) ** 2)  # Variance of the differences in the column

    information_loss = np.sum(calculate_information_loss(x, y, S)) / (p * n)
    total_information_loss += information_loss

print("Total Information Loss:", total_information_loss)
